# GEWDiff-BTP Baseline Analysis
**Project**: Diffusion-based Hyperspectral Image Super-Resolution  
**Baseline Reference**: epoch_200.pth on WDC dataset  
**Metrics**: MPSNR=33.7827, MSSIM=0.6920, SAM=9.1510  

This notebook analyzes and reproduces the baseline GEWDiff results locally in VS Code.

In [7]:
import sys
import os
from pathlib import Path
import numpy as np

# ===== PROJECT PATHS =====
ROOT = Path.cwd().parent
GEWDIFF = ROOT / "src" / "GEWDiff"
DATA = GEWDIFF / "data"
MODEL = GEWDIFF / "model"
UTILS = GEWDIFF / "utils"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS = ROOT / "results"

# Create results directory if it doesn't exist
RESULTS.mkdir(exist_ok=True)

# ===== BASELINE CONFIGURATION =====
BASELINE_CHECKPOINT = CHECKPOINTS / "epoch_200.pth"
BASELINE_METRICS = {
    "MPSNR": 33.7827,
    "MSSIM": 0.6920,
    "SAM": 9.1510,
    "CC": 0.6275,
    "RMSE": 0.05695,
    "FID": 47.8211,
    "LV_Pred": 0.005330,
    "LV_True": 0.007440,
}

# ===== SETUP PYTHON PATH =====
sys.path.insert(0, str(GEWDIFF))
sys.path.insert(0, str(DATA))
sys.path.insert(0, str(MODEL))
sys.path.insert(0, str(UTILS))

# ===== VERIFY PATHS =====
print("=" * 60)
print("GEWDiff-BTP Project Setup")
print("=" * 60)
print(f"\n📁 Project Root:        {ROOT}")
print(f"📁 GEWDiff Source:      {GEWDIFF}")
print(f"📁 Data Directory:      {DATA}")
print(f"📁 Model Directory:     {MODEL}")
print(f"📁 Checkpoints:         {CHECKPOINTS}")
print(f"📁 Results:             {RESULTS}")

print(f"\n✓ GEWDiff exists:       {GEWDIFF.exists()}")
print(f"✓ Data directory:       {DATA.exists()}")
print(f"✓ Baseline checkpoint:  {BASELINE_CHECKPOINT.exists()}")

if BASELINE_CHECKPOINT.exists():
    size_gb = BASELINE_CHECKPOINT.stat().st_size / (1024**3)
    print(f"  └─ Size: {size_gb:.2f} GB")

# ===== ENVIRONMENT INFO =====
print(f"\n🐍 Python Version:      {sys.version.split()[0]}")
print(f"📦 NumPy Version:       {np.__version__}")

try:
    import torch
    print(f"🔥 PyTorch Version:     {torch.__version__}")
    print(f"   CUDA Available:     {torch.cuda.is_available()}")
except ImportError:
    print("⚠ PyTorch not imported")

print("\n" + "=" * 60)
print("✅ SETUP COMPLETE")
print("=" * 60)

GEWDiff-BTP Project Setup

📁 Project Root:        c:\Projects\BTP-GEWDiff\GEWDiff-BTP
📁 GEWDiff Source:      c:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff
📁 Data Directory:      c:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff\data
📁 Model Directory:     c:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff\model
📁 Checkpoints:         c:\Projects\BTP-GEWDiff\GEWDiff-BTP\checkpoints
📁 Results:             c:\Projects\BTP-GEWDiff\GEWDiff-BTP\results

✓ GEWDiff exists:       True
✓ Data directory:       True
✓ Baseline checkpoint:  False

🐍 Python Version:      3.10.19
📦 NumPy Version:       1.26.4
🔥 PyTorch Version:     2.13.0+cpu
   CUDA Available:     False

✅ SETUP COMPLETE


In [11]:
# ===== IMPORT GEWDiff MODULES =====
print("\n📦 Importing GEWDiff modules...\n")

# Core data processing modules
try:
    import dataset
    print("✓ dataset.py (core data processing)")
except ImportError as e:
    print(f"✗ dataset.py: {e}")

try:
    from RWT import rwa, inv_rwa
    print("✓ RWT.py (wavelet transforms: rwa, inv_rwa)")
except ImportError as e:
    print(f"✗ RWT.py: {e}")

try:
    from eval import quality_assessment, compare_sam, compare_ergas, compare_mpsnr, compare_mssim
    print("✓ eval.py (metrics: quality_assessment, MPSNR, MSSIM, SAM, ERGAS)")
except ImportError as e:
    print(f"✗ eval.py: {e}")

# Model modules (may have complex relative imports - will be loaded via training/inference scripts)
print("\n📚 Model modules (complex dependencies, will be loaded via train/test scripts):\n")

# Check if files exist
import os
model_files = {
    "unet3d.py": os.path.exists(MODEL / "unet3d.py"),
    "edm.py": os.path.exists(MODEL / "edm.py"),
    "RWT.py": os.path.exists(MODEL / "RWT.py"),
}

for filename, exists in model_files.items():
    status = "✓ exists" if exists else "✗ missing"
    print(f"  {status}: {filename}")

# Standard dependencies
print("\n📚 Standard library dependencies...\n")
import torch
import torchvision
from torchvision import transforms
import scipy
from scipy.ndimage import zoom
from sklearn.decomposition import PCA
from skimage import exposure
import tifffile
import pywt
import imageio

print("✓ torch, torchvision, transforms")
print("✓ scipy, scipy.ndimage.zoom")
print("✓ sklearn.decomposition.PCA")
print("✓ skimage.exposure")
print("✓ tifffile")
print("✓ pywt (PyWavelets)")
print("✓ imageio")

print("\n" + "=" * 60)
print("✅ CORE MODULES LOADED & READY")
print("=" * 60)
print("\n📝 Next: Load baseline checkpoint and inspect inference pipeline")


📦 Importing GEWDiff modules...

✓ dataset.py (core data processing)
✓ RWT.py (wavelet transforms: rwa, inv_rwa)
✓ eval.py (metrics: quality_assessment, MPSNR, MSSIM, SAM, ERGAS)

📚 Model modules (complex dependencies, will be loaded via train/test scripts):

  ✓ exists: unet3d.py
  ✓ exists: edm.py
  ✓ exists: RWT.py

📚 Standard library dependencies...

✓ torch, torchvision, transforms
✓ scipy, scipy.ndimage.zoom
✓ sklearn.decomposition.PCA
✓ skimage.exposure
✓ tifffile
✓ pywt (PyWavelets)
✓ imageio

✅ CORE MODULES LOADED & READY

📝 Next: Load baseline checkpoint and inspect inference pipeline


## Section 1: Baseline Data & Configuration

Below we explore the WDC test data structure and verify the baseline configuration.

In [12]:
# ===== EXPLORE WDC TEST DATA =====
print("\n📊 WDC Test Data Structure")
print("=" * 60)

wdc_path = DATA / "test_wdc"
print(f"\nWDC Test Path: {wdc_path}")
print(f"Exists: {wdc_path.exists()}\n")

if wdc_path.exists():
    # List subdirectories
    subdirs = [d for d in wdc_path.iterdir() if d.is_dir()]
    print(f"Subdirectories ({len(subdirs)}):")
    for subdir in sorted(subdirs):
        files = list(subdir.glob("*"))
        print(f"  📁 {subdir.name}/")
        for f in sorted(files):
            size_str = f"{f.stat().st_size / (1024**2):.1f} MB" if f.is_file() else ""
            print(f"     └─ {f.name} {size_str}")

# ===== BASELINE CONFIGURATION =====
print("\n\n⚙️  Baseline Configuration (from test_wdc_local.py)")
print("=" * 60)

config = {
    "compack_bands": 121,
    "pca_bands": 20,
    "num_epochs": 200,
    "timesteps": 50,
    "train_batch_size": 1,
    "mask": True,
    "edge": True,
    "l1_lambda": 0.8,
    "l2_lambda": 0.1,
    "l3_lambda": 0.1,
    "sigma_min": 0.002,
    "sigma_max": 80,
    "sigma_data": 0.5,
    "rho": 7,
}

print("\nHyperspectral Processing:")
print(f"  • Original bands: {config['compack_bands']}")
print(f"  • PCA compressed: {config['pca_bands']}")
print(f"  • Compression ratio: {config['compack_bands'] / config['pca_bands']:.1f}x")

print("\nDiffusion Sampling:")
print(f"  • Timesteps: {config['timesteps']}")
print(f"  • Sigma min: {config['sigma_min']}")
print(f"  • Sigma max: {config['sigma_max']}")
print(f"  • Sigma data: {config['sigma_data']}")
print(f"  • Rho (schedule): {config['rho']}")

print("\nTraining Checkp oint:")
print(f"  • Epochs: {config['num_epochs']}")
print(f"  • Batch size: {config['train_batch_size']}")

print("\nLoss Weights:")
print(f"  • L1 (reconstruction): {config['l1_lambda']}")
print(f"  • L2 (spectral): {config['l2_lambda']}")
print(f"  • L3 (edge): {config['l3_lambda']}")

print("\nAuxiliary Processing:")
print(f"  • Mask: {config['mask']}")
print(f"  • Edge: {config['edge']}")

print("\n\n📈 BASELINE METRICS (Reference)")
print("=" * 60)
for metric, value in BASELINE_METRICS.items():
    print(f"  {metric:.<20} {value:.6f}")


📊 WDC Test Data Structure

WDC Test Path: c:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff\data\test_wdc
Exists: False



⚙️  Baseline Configuration (from test_wdc_local.py)

Hyperspectral Processing:
  • Original bands: 121
  • PCA compressed: 20
  • Compression ratio: 6.0x

Diffusion Sampling:
  • Timesteps: 50
  • Sigma min: 0.002
  • Sigma max: 80
  • Sigma data: 0.5
  • Rho (schedule): 7

Training Checkp oint:
  • Epochs: 200
  • Batch size: 1

Loss Weights:
  • L1 (reconstruction): 0.8
  • L2 (spectral): 0.1
  • L3 (edge): 0.1

Auxiliary Processing:
  • Mask: True
  • Edge: True


📈 BASELINE METRICS (Reference)
  MPSNR............... 33.782700
  MSSIM............... 0.692000
  SAM................. 9.151000
  CC.................. 0.627500
  RMSE................ 0.056950
  FID................. 47.821100
  LV_Pred............. 0.005330
  LV_True............. 0.007440


In [9]:
import sys

# Install missing packages
print("Installing missing dependencies...")
!{sys.executable} -m pip install diffusers einops -q
print("✓ diffusers and einops installed")

Installing missing dependencies...
✓ diffusers and einops installed


In [2]:
pip install torch

  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/122.0 MB ? eta -:--:--
    --------------------------------------- 2.4/122.0 MB 16.8 MB/s eta 0:00:08
   -- ------------------------------------- 6.3/122.0 MB 16.8 MB/s eta 0:00:07
   --- ------------------------------------ 10.2/122.0 MB 17.7 MB/s eta 0:00:07
   ---- ----------------------------------- 13.6/122.0 MB 17.5 MB/s eta 0:00:07
   ----- ---------------------------------- 17.0/122.0 MB 16.8 MB/s eta 0:00:07
   ------ --------------------------------- 20.4/122.0 MB 16.8 MB/s eta 0:00:07
   -------- ------------------------------- 24.9/122.0 MB 17.3 MB/s eta 0:00:06
   --------- ------------------------------ 29.9/1

In [4]:
import sys
import torch
import numpy as np

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__)


Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:23:22) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.13.0+cpu
CUDA available: False
NumPy: 1.26.4


In [12]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
GEWDIFF = ROOT / "src" / "GEWDiff"

sys.path.insert(0, str(GEWDIFF))
sys.path.insert(0, str(GEWDIFF / "data"))

print("Project root:", ROOT)
print("GEWDiff exists:", GEWDIFF.exists())
print("Dataset exists:", (GEWDIFF / "data" / "dataset.py").exists())

import dataset

print("dataset import: OK")

Project root: c:\Projects\BTP-GEWDiff\GEWDiff-BTP
GEWDiff exists: True
Dataset exists: True


ModuleNotFoundError: No module named 'tifffile'

In [8]:
import sys
import subprocess

print("Python executable:")
print(sys.executable)

print("\nPip:")
subprocess.run([sys.executable, "-m", "pip", "--version"])

Python executable:
c:\Users\heman\anaconda3\envs\ml_stable\python.exe

Pip:


CompletedProcess(args=['c:\\Users\\heman\\anaconda3\\envs\\ml_stable\\python.exe', '-m', 'pip', '--version'], returncode=0)

In [9]:
import sys
!{sys.executable} -m pip install torchvision

   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   -------------- ------------------------- 1.3/3.5 MB 9.6 MB/s eta 0:00:01
   -------------------------------------- - 3.4/3.5 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 8.6 MB/s  0:00:00


In [10]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.13.0+cpu
Torchvision: 0.28.0+cpu


In [14]:
import dataset
print("dataset import: OK")

ModuleNotFoundError: No module named 'RWT'

In [3]:
import sys
!{sys.executable} -m pip install tifffile scipy

In [15]:
sys.path.insert(0, str(GEWDIFF / "model"))

In [16]:
import dataset
print("✅ dataset import: OK")

ModuleNotFoundError: No module named 'pywt'

In [17]:
import sys

!{sys.executable} -m pip install \
    -r "../src/GEWDiff/requirements.txt" \
    --no-deps

  Using cached importlib_metadata-9.0.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)


ERROR: Ignored the following yanked versions: 3.4.11.39, 3.4.17.61, 4.4.0.42, 4.4.0.44, 4.5.4.58, 4.5.5.62, 4.7.0.68
ERROR: Ignored the following versions that require a different python version: 1.3.3 Requires-Python >=3.11; 2.4.0a0 Requires-Python >=3.11; 2.4.0a1 Requires-Python >=3.11; 3.11.0 Requires-Python >=3.11; 3.11.0rc1 Requires-Python >=3.11; 3.11.0rc2 Requires-Python >=3.11; 3.11.1 Requires-Python >=3.11; 3.13.0 Requires-Python >=3.11; 3.13.1 Requires-Python >=3.11; 3.13.2 Requires-Python >=3.11; 3.14.0 Requires-Python >=3.11; 3.14.1 Requires-Python >=3.11; 3.15.0 Requires-Python >=3.11; 3.15.1 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement opencv-python==4.10.0 (from versions: 3.4.0.14, 3.4.10.37, 3.4.11.41, 3.4.11.43, 3.4.11.45, 3.4.13.47, 3.4.15.55, 3.4.16.57, 3.4.16.59, 3.4.17.63, 3.4.18.65, 4.3.0.38, 4.4.0.40, 4.4.0.46, 4.5.1.48, 4.5.3.56, 4.5.4.60, 4.5.5.64, 4.6.0.66, 4.7.0.72, 4.8.0.74, 4.8.0.76, 4.8.1.78, 4.9.0.80, 4.10.0.82, 4.

In [18]:
from pathlib import Path

p = Path("../src/GEWDiff/requirements_vscode.txt")
text = p.read_text()

text = text.replace("opencv-python==4.10.0", "opencv-python==4.10.0.84")
text = text.replace("opencv-python-headless==4.10.0", "opencv-python-headless==4.10.0.84")

p.write_text(text)

print("✅ OpenCV versions fixed")

FileNotFoundError: [Errno 2] No such file or directory: '..\\src\\GEWDiff\\requirements_vscode.txt'

In [19]:
from pathlib import Path

matches = list(Path(r"C:\Projects\BTP-GEWDiff\GEWDiff-BTP").rglob("requirements_vscode.txt"))
print(matches)

[]


In [20]:
p = Path(r"C:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff\requirements_vscode.txt")

print(p.exists())

False


In [21]:
text = p.read_text()

text = text.replace("opencv-python==4.10.0", "opencv-python==4.10.0.84")
text = text.replace("opencv-python-headless==4.10.0", "opencv-python-headless==4.10.0.84")

p.write_text(text)

print("✅ Fixed")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Projects\\BTP-GEWDiff\\GEWDiff-BTP\\src\\GEWDiff\\requirements_vscode.txt'

In [22]:
from pathlib import Path

repo = Path(r"C:\Projects\BTP-GEWDiff\GEWDiff-BTP")
src = repo / "src" / "GEWDiff"

original = src / "requirements.txt"
vscode = src / "requirements_vscode.txt"

text = original.read_text()

# Keep the currently working PyTorch stack
remove = [
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
    "triton==3.1.0",
]

lines = [line for line in text.splitlines() if line.strip() not in remove]

text = "\n".join(lines)
text = text.replace("opencv-python==4.10.0", "opencv-python==4.10.0.84")
text = text.replace("opencv-python-headless==4.10.0", "opencv-python-headless==4.10.0.84")

vscode.write_text(text + "\n")

print("✅ Created:", vscode)
print("Exists:", vscode.exists())

✅ Created: C:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff\requirements_vscode.txt
Exists: True


In [23]:
import sys

!{sys.executable} -m pip install -r "C:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff\requirements_vscode.txt"

  Using cached absl_py-2.2.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached brotli-1.2.0-cp310-cp310-win_amd64.whl.metadata (6.3 kB)
  Using cached contourpy-1.3.1-cp310-cp310-win_amd64.whl.metadata (5.4 kB)
  Using cached diffusers-0.39.0-py3-none-any.whl.metadata (20 kB)
  Using cached einops-0.8.0-py3-none-any.whl.metadata (12 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached fonttools-4.55.3-cp310-cp310-win_amd64.whl.metadata (168 kB)
  Using cached gmpy2-2.3.1-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached grpcio-1.71.0-cp310-cp310-win_amd64.whl.metadata (4.0 kB)
  Using cached h5py-3.12.1-cp310-cp310-win_amd64.whl.metadata (2.5 kB)
  Using cached huggingface_hub-1.28.0-py3-none-any.whl.metadata (16 kB)
  Using cached imageio-2.37.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached importlib_metadata-9.0.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached

ERROR: Ignored the following versions that require a different python version: 0.23.0 Requires-Python >=3.6,<3.10; 0.26.0 Requires-Python >=3.11; 0.26.0rc1 Requires-Python >=3.11; 0.26.0rc2 Requires-Python >=3.11; 1.16.0 Requires-Python >=3.11; 1.16.0rc1 Requires-Python >=3.11; 1.16.0rc2 Requires-Python >=3.11; 1.16.1 Requires-Python >=3.11; 1.16.2 Requires-Python >=3.11; 1.16.3 Requires-Python >=3.11; 1.17.0 Requires-Python >=3.11; 1.17.0rc1 Requires-Python >=3.11; 1.17.0rc2 Requires-Python >=3.11; 1.17.1 Requires-Python >=3.11; 1.18.0 Requires-Python >=3.12; 1.18.0rc1 Requires-Python >=3.12; 1.18.0rc2 Requires-Python >=3.12; 1.3.3 Requires-Python >=3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.8.0 Requires-Python >=3.11; 1.8.0rc1 Requires-Python >=3.11; 1.9.0 Requires-Python >=3.11; 1.9.0rc1 Requires-Python >=3.11; 2.4.0a0 Requires-Python >=3.11; 2.4.0a1 Requires-Python >=3.11; 3.11.

In [5]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Projects\BTP-GEWDiff\GEWDiff-BTP")
GEWDIFF = ROOT / "src" / "GEWDiff"

sys.path.insert(0, str(GEWDIFF / "data"))
sys.path.insert(0, str(GEWDIFF / "model"))

print("GEWDiff:", GEWDIFF)
print("dataset.py:", (GEWDIFF / "data" / "dataset.py").exists())
print("RWT.py:", (GEWDIFF / "model" / "RWT.py").exists())

GEWDiff: C:\Projects\BTP-GEWDiff\GEWDiff-BTP\src\GEWDiff
dataset.py: True
RWT.py: True


In [6]:
import dataset
print("✅ GEWDiff dataset import: OK")

✅ GEWDiff dataset import: OK


In [4]:
import sys
!{sys.executable} -m pip install PyWavelets

  Using cached pywavelets-1.8.0-cp310-cp310-win_amd64.whl.metadata (9.0 kB)
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.2 MB 4.2 MB/s eta 0:00:01
   ----------------- ---------------------- 1.8/4.2 MB 4.8 MB/s eta 0:00:01
   ------------------------------------- -- 3.9/4.2 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 6.3 MB/s  0:00:00


In [5]:
import pywt
print("PyWavelets:", pywt.__version__)

PyWavelets: 1.8.0


In [4]:
import sys
!{sys.executable} -m pip install scikit-image sewar

  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.8 MB 14.7 MB/s eta 0:00:01
   ---------------- ----------------------- 5.2/12.8 MB 13.8 MB/s eta 0:00:01
   ------------------------ --------------- 7.9/12.8 MB 13.1 MB/s eta 0:00:01
   ------------------------------- -------- 10.2/12.8 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.8 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 12.2 MB/s  0:00:01
Using cached lazy_loader-0.5-py3-none-any.whl (8.0 kB)

   ---------- ----------------------------- 1/4 [imageio]
   ---------- ----------------------------- 1/4 [imageio]
   ---------- ----------------------------- 1/4 [imageio]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ ------